# DREME Commit 2 | SOI 220 nm coupled supermode smoke test

Run **1A Settings and geometry**, then **1B MODE API check**. The agreed platform is OTIP SOI with a 220 nm silicon device layer. Widths, wavelength, gap, stack details, domain, mesh and mode counts are editable smoke-test examples. The default example is a fully etched film between oxide claddings.

Choose one workflow in 1A, review the settings, set `CONFIRM_FDE_SETTINGS=True`, and rerun 1A.

| Workflow | Run after 1A and 1B |
| --- | --- |
| `"files"` | **2 Prepare external MODE jobs** → manually calculate modes and export each file in MODE → **3 Import external MODE results** → **4 Reopen and propagate offline** |
| `"sweep"` | **2 Prepare external MODE jobs** → manually run each prepared Sweep in MODE → **3 Import external MODE results** → **4 Reopen and propagate offline** |
| `"python"` | **2P Acquire coupled FDE** runs the Python loop → **4 Reopen and propagate offline** |

Avoid **Run All**. The external preparation and import cells use the MODE API to save or read projects. All solves for these two workflows are started manually in the MODE UI. The layout comes from `StraightGapSectionModel` and the adapter, without a separate LMS template.

Select a Python kernel with the repository dependencies installed. Start Jupyter in the checkout or a directory inside it. If needed, set DREME_REPO to the checkout and LUMERICAL_API_DIR to the directory containing lumapi.py, or edit REPO and MODE_API_DIR in 1A.

[Ansys FDE settings](https://optics.ansys.com/hc/en-us/articles/360034916973-FDE-solver-Simulation-object) · [Ansys parameter Sweep](https://optics.ansys.com/hc/en-us/articles/360034922893-Creating-parameter-sweeps-using-a-script)

## 1A Settings and geometry

Inspect the cross sections for all gaps and the printed settings. Length variables use the units in their names. Confirm material names against your MODE material library, and set a new `RUN_NAME` for each acquisition. `CLADDING_INDEX` controls DREME's guided-mode classification, while the FDE stack uses the named materials.

In [ ]:
# 1A Settings and geometry -- no MODE session or solve
from pathlib import Path
import importlib
import json
import os
import sys
import time
import traceback

REPO = os.environ.get("DREME_REPO")              # Auto-detect the checkout if unset.
MODE_API_DIR = os.environ.get("LUMERICAL_API_DIR")  # Use the DREME configuration if unset.
MODE_SHOW_WINDOW = True                  # Visible API sessions connected on the local MODE 2025 R1 test.
OUTPUT_ROOT = Path.home() / "DREME" / "coupled_waveguide_runs"
RUN_NAME = "soi220_external_01"
MODE_WORKFLOW = "files"                   # "files", "sweep", or "python".
SWEEP_BATCHES_NM = None                  # None = one Sweep with all gaps.
# Example split: ((150, 250), (350,)); the union must cover GAPS_NM before import.
JOBS_ROOT = OUTPUT_ROOT                  # Change to prepare projects on another drive.
CONFIRM_FDE_SETTINGS = False

# Editable examples. Only DEVICE_LAYER_NM is fixed by the agreed SOI platform.
DEVICE_LAYER_NM = 220
WG0_WIDTH_NM, WG1_WIDTH_NM, WG1_CENTER_Y_NM = 500, 450, 0
WAVELENGTH_NM = 1550
GAPS_NM = (150, 250, 350)                 # Choose exactly 2 or 3 increasing gaps.
SLAB_NM, DEVICE_LAYER_BOTTOM_Z_NM = 0, 0
BOX_THICKNESS_UM = None                   # e.g. 2.0 if including a finite BOX.
SUBSTRATE_MATERIAL_ID = None             # e.g. "si", paired with BOX_THICKNESS_UM.
CORE_MATERIAL_ID, UPPER_MATERIAL_ID, LOWER_MATERIAL_ID = "si", "oxide", "oxide"
MATERIAL_BINDINGS = {"si": "Si (Silicon) - Palik", "oxide": "SiO2 (Glass) - Palik"}
CLADDING_INDEX = 1.44
Y_BOUNDS_UM, Z_BOUNDS_UM = (-2.0, 2.4), (-1.3, 1.3)
MESH_CELLS_YZ = (176, 104)
BOUNDARY = "Metal"
CANDIDATE_COUNTS = {"coupled": 2, "wg0": 2, "wg1": 2}  # Isolated budgets are stored for later use.
OFFLINE_LENGTH_UM = 10
PML_MODE_SINK = True                      # Existing DREME default; no accuracy claim here.

def is_checkout(path):
    return (path / "em_simulation").is_dir() and (path / "config" / "config.yaml").is_file()

cwd = Path.cwd().resolve()
REPO = (Path(REPO).expanduser().resolve() if REPO else
        next((path for path in (cwd, *cwd.parents) if is_checkout(path)), None))
if REPO is None or not is_checkout(REPO):
    raise FileNotFoundError("Set REPO in 1A or DREME_REPO to the DREME checkout.")
MODE_API_DIR = Path(MODE_API_DIR).expanduser().resolve() if MODE_API_DIR else None
if sys.path[:1] != [str(REPO)]:
    sys.path.insert(0, str(REPO))
if MODE_API_DIR is not None and MODE_API_DIR.is_dir() and sys.path[:1] != [str(MODE_API_DIR)]:
    sys.path.insert(0, str(MODE_API_DIR))

import matplotlib.pyplot as plt
import numpy as np
import em_simulation as dreme
from em_simulation.data_updater.lumerical_section import validate_section
from em_simulation.geometry.coupled_waveguide.cross_section import plot_cross_section

assert Path(dreme.__file__).resolve().parent.parent == REPO.resolve(), "Restart the kernel: a different DREME is imported"
if len(GAPS_NM) not in (2, 3):
    raise ValueError("Choose exactly 2 or 3 gap nodes for this notebook")
if not RUN_NAME or RUN_NAME in (".", "..") or Path(RUN_NAME).name != RUN_NAME:
    raise ValueError("RUN_NAME must be a single directory name")

if MODE_WORKFLOW not in ("files", "sweep", "python"):
    raise ValueError("MODE_WORKFLOW must be 'files', 'sweep', or 'python'")
DATASET_DIR = OUTPUT_ROOT / RUN_NAME
JOBS_DIR = JOBS_ROOT / f"{RUN_NAME}_{MODE_WORKFLOW}_jobs"
JOBS_PLAN = JOBS_DIR / "external_fde_jobs.json"
IMPORT_LOG = OUTPUT_ROOT / f"{RUN_NAME}.import.json"
ACQUISITION_LOG = OUTPUT_ROOT / f"{RUN_NAME}.acquisition.json"
OFFLINE_LOG = OUTPUT_ROOT / f"{RUN_NAME}.offline.json"

stack = dreme.WaveguideStack(
    thickness=DEVICE_LAYER_NM * 1e-9,
    core_material_id=CORE_MATERIAL_ID,
    upper_cladding_material_id=UPPER_MATERIAL_ID,
    lower_cladding_material_id=LOWER_MATERIAL_ID,
    device_layer_bottom_z=DEVICE_LAYER_BOTTOM_Z_NM * 1e-9,
    slab_thickness=SLAB_NM * 1e-9,
    box_thickness=None if BOX_THICKNESS_UM is None else BOX_THICKNESS_UM * 1e-6,
    substrate_material_id=SUBSTRATE_MATERIAL_ID,
)
section_model = dreme.StraightGapSectionModel(
    wg0_width=WG0_WIDTH_NM * 1e-9, wg1_width=WG1_WIDTH_NM * 1e-9,
    wg1_center_y=WG1_CENTER_Y_NM * 1e-9, stack=stack,
)
settings = dreme.CoupledDatasetSettings(
    gap_grid=tuple(g * 1e-9 for g in GAPS_NM), cladding_index=CLADDING_INDEX,
    fde=dreme.FDEConfig(
        wavelength=WAVELENGTH_NM * 1e-9,
        y_bounds=tuple(v * 1e-6 for v in Y_BOUNDS_UM),
        z_bounds=tuple(v * 1e-6 for v in Z_BOUNDS_UM),
        mesh_cells_yz=MESH_CELLS_YZ, material_bindings=MATERIAL_BINDINGS,
        candidate_counts=CANDIDATE_COUNTS, boundary=BOUNDARY,
    ),
)
print(f"Python {sys.version.split()[0]} | {sys.executable}")
print(f"DREME: {Path(dreme.__file__).resolve()}")
print(f"Output: {DATASET_DIR} (already exists: {DATASET_DIR.exists()})")
print(f"Workflow: {MODE_WORKFLOW} | External projects: {JOBS_DIR}")
print(f"Logs: {ACQUISITION_LOG}, {IMPORT_LOG}, and {OFFLINE_LOG}")
print(json.dumps({"dataset": settings.configuration(), "section": section_model.configuration()}, indent=2))

fig, axes = plt.subplots(1, len(settings.gap_grid), figsize=(4 * len(settings.gap_grid), 3))
for ax, gap in zip(axes, settings.gap_grid):
    section = section_model.describe_cross_section({"gap": gap})
    validate_section(section, settings.fde)
    plot_cross_section(section, ax)
    ax.set(xlim=settings.fde.y_bounds, ylim=settings.fde.z_bounds, title=f"{gap * 1e9:g} nm gap")
plt.tight_layout()
plt.show()

## 1B MODE API check

This imports the API module and checks its path. MODE_SHOW_WINDOW in 1A controls the sessions opened by later cells. It does not instantiate `MODE`, verify a license, or solve. When MODE_API_DIR is unset, DREME uses the current Python path and its config/config.yaml search paths. If you change MODE_API_DIR, restart the kernel and run 1A and 1B again.

In [ ]:
# 1B MODE API check -- no MODE session or solve
expected_api = MODE_API_DIR / "lumapi.py" if MODE_API_DIR is not None else None
if expected_api is not None and not expected_api.is_file():
    raise FileNotFoundError(f"Set MODE_API_DIR to the directory containing lumapi.py: {MODE_API_DIR}")
try:
    mode_api = importlib.import_module("lumapi")
except (ImportError, OSError) as exc:
    raise RuntimeError("Could not import lumapi. Set MODE_API_DIR to the directory containing "
                       "lumapi.py and restart the kernel.") from exc
loaded_api = Path(mode_api.__file__).resolve()
if ((expected_api is not None and loaded_api != expected_api.resolve())
        or not callable(getattr(mode_api, "MODE", None))):
    raise RuntimeError(f"Unexpected MODE API module: {loaded_api}. Restart the kernel and check MODE_API_DIR.")
if MODE_SHOW_WINDOW:
    from types import SimpleNamespace
    imported_mode_api = mode_api
    mode_api = SimpleNamespace(MODE=lambda hide=True: imported_mode_api.MODE(hide=False))
print(f"MODE API import OK: {loaded_api}")
print("Layout recipe: new 2D X-normal MODE FDE, generated from the section polygons and settings above.")

## 2 Prepare external MODE jobs

This opens MODE through the API to write layouts and check the polygons at each configured gap. It does not call Calculate Modes or Run Sweep. `JOBS_PLAN` records every project, gap, export script, expected result file, and the fixed settings.

For multiple machines or folders, keep each `external_fde_jobs.json` alongside its results. Keep the full target `GAPS_NM` in 1A, and use `SWEEP_BATCHES_NM` to select subsets. Choose a different preparation folder for each job set. Separately prepared plans can contain different gap lists if their physical settings match. All selected result files together must cover every target gap once.

In [ ]:
# 2 Prepare external MODE jobs -- creates layouts, no solve
from em_simulation.data_updater.external_mode import ModeResult, prepare_mode_jobs

if MODE_WORKFLOW not in ("files", "sweep") or CONFIRM_FDE_SETTINGS is not True:
    raise RuntimeError("Choose an external workflow and confirm the 1A settings first.")
batches_m = None
if MODE_WORKFLOW == "sweep" and SWEEP_BATCHES_NM is not None:
    batches_m = [tuple(g * 1e-9 for g in batch) for batch in SWEEP_BATCHES_NM]

def external_geometry():
    updater = dreme.DataUpdater(DATASET_DIR, coupled_settings=settings)
    return dreme.StraightGapCoupler(
        updater, section_model,
        gap_profile=dreme.profile_from_samples(
            [0.0, OFFLINE_LENGTH_UM * 1e-6], [settings.gap_grid[0], settings.gap_grid[-1]],
        ),
        verbose=False,
    )

plan = prepare_mode_jobs(external_geometry(), JOBS_DIR, mode_api=mode_api,
                         workflow=MODE_WORKFLOW, batches=batches_m)
print(f"Prepared without solves: {plan}")
for job in json.loads(plan.read_text(encoding="utf-8"))["jobs"]:
    print(f"Open {plan.parent / job['project']}")
    for point in job["points"]:
        print(f"  {point['gap_m'] * 1e9:g} nm | export: {point['export_script']} -> {point['result_ldf']}")

## In the MODE UI

**Individual files.** Open each listed `.lms`. In **Eigensolver Analysis → Modal Analysis**, click **Calculate Modes**. With the solved mode cards still present, open the matching `*_export.lsf` in **Script File Editor** and click **Run Script**. It exports one `.ldf` with the gap, job ID, layout and analysis settings, and all requested cards' E/H, coordinates, neff and TE fraction. It does not solve. If the project folder moved, update dreme_export_path at the top of the script. Keep the preparation plan and the exported files. A solved `.lms` with its local mode cards still readable is also accepted. [Ansys MODE modal analysis](https://optics.ansys.com/hc/en-us/articles/360034917353-FDE-solver-analysis-Modal-Analysis-Tab) · [Ansys savedata](https://optics.ansys.com/hc/en-us/articles/360034411174-savedata-Script-command)

**Native Sweep.** Open each parent `.lms`. In **Optimizations and Sweeps**, select **dreme_gap_sweep** and click **Run** (or right-click **Run**). After completion save the parent at the same path and keep its sweep results directory and solved child projects. The registered parameter is `dreme_gap::gap`. Only the bus polygon follows it. [Ansys Sweep setup and results](https://optics.ansys.com/hc/en-us/articles/360034922893-Creating-parameter-sweeps-using-a-script)

The aggregate reader expects explicit `gap` on each result, all registered E/H, neff, TE and x/y/z results, and E/H order **y, z, gap, component** after dropping singleton axes. If MODE cannot aggregate this format, use the solved child `.lms` files as **node** inputs. If local mode cards are unavailable after saving, use the matching prepared export script while that solved child's cards are still present, and import its `.ldf`. In the local MODE 2025 R1 verification, the native Sweep retained aggregate results, but reopening its child projects did not retain the mode cards. If that happens, use the aggregate results or solve individual projects and export their cards before closing them. Import never re-solves. The filenames are only for locating files. Import reads the actual gap and job ID.

## 3 Import external MODE results

Supply result paths explicitly in `MODE_RESULTS` for renamed files, saved child projects, separate preparation folders, or mixed sources. The default uses paths listed in the local plan. The import uses saved results without calling Calculate Modes or Run Sweep. It creates the same canonical dataset as the Python workflow, including edges across job boundaries.

In [ ]:
# 3 Import external MODE results -- reads saved results, no solve
from em_simulation.data_updater.external_mode import ModeResult

# Each entry is ModeResult(result_path, preparation_plan_path, kind="node" or "sweep").
MODE_RESULTS = None
# MODE_RESULTS = [
#     ModeResult(JOBS_DIR / "files_001_node_001.ldf", JOBS_PLAN),
#     ModeResult(JOBS_ROOT / "other_jobs" / "solved_child.lms",
#                JOBS_ROOT / "other_jobs" / "external_fde_jobs.json", kind="node"),
#     ModeResult(JOBS_ROOT / "sweep_batch" / "sweep_001.lms",
#                JOBS_ROOT / "sweep_batch" / "external_fde_jobs.json", kind="sweep"),
# ]

if MODE_WORKFLOW not in ("files", "sweep"):
    raise RuntimeError("This cell is for externally solved files or Sweep results.")
if MODE_RESULTS is None:
    jobs = json.loads(JOBS_PLAN.read_text(encoding="utf-8"))["jobs"]
    MODE_RESULTS = [
        item
        for job in jobs
        for item in (
            [ModeResult(JOBS_PLAN.parent / job["project"], JOBS_PLAN, "sweep")]
            if job["workflow"] == "sweep" else
            [ModeResult(JOBS_PLAN.parent / p["result_ldf"], JOBS_PLAN) for p in job["points"]]
        )
    ]

started = time.perf_counter()
report = {"cell": "3 Import external MODE results", "dataset": str(DATASET_DIR),
          "sources": [str(item.path) for item in MODE_RESULTS]}
try:
    updater = dreme.DataUpdater(DATASET_DIR, coupled_settings=settings)
    geometry = dreme.StraightGapCoupler(
        updater, section_model,
        gap_profile=dreme.profile_from_samples(
            [0.0, OFFLINE_LENGTH_UM * 1e-6], [settings.gap_grid[0], settings.gap_grid[-1]],
        ),
        verbose=False,
    )
    manifest = updater.import_coupled(geometry, MODE_RESULTS, mode_api=mode_api)
    report.update(status="complete", generation_id=manifest["generation_id"],
                  gaps_nm=[entry["node"][0] * 1e9 for entry in manifest["nodes"]],
                  directed_edges=len(manifest["edges"]))
    print(f"Saved {DATASET_DIR}: {report['gaps_nm']} nm, {report['directed_edges']} directed edges")
except Exception as exc:
    report.update(status="error", first_error=f"{type(exc).__name__}: {exc}",
                  traceback=traceback.format_exc())
    raise
finally:
    report["elapsed_s"] = time.perf_counter() - started
    OUTPUT_ROOT.mkdir(parents=True, exist_ok=True)
    IMPORT_LOG.write_text(json.dumps(report, indent=2), encoding="utf-8")
    print(f"{report['status']} | {IMPORT_LOG}")

## 2P Acquire coupled FDE (Python)

**Run this cell manually to start real FDE.** A success line per node reports the requested/returned mode counts and E/H shape. The generation publishes `canonical_supermodes.json` after the three modal pickle files are complete. The adjacent acquisition JSON records the first failure and traceback if a solve, export, or save fails. Use a fresh run name to retry.

In [ ]:
# 2P Acquire coupled FDE -- real MODE opens and findmodes runs here
if MODE_WORKFLOW != "python":
    raise RuntimeError("This cell is for MODE_WORKFLOW='python'. External workflows use 2 and 3.")
if CONFIRM_FDE_SETTINGS is not True:
    raise RuntimeError("Review 1A, set CONFIRM_FDE_SETTINGS=True, and rerun 1A before starting MODE.")
if DATASET_DIR.exists() or ACQUISITION_LOG.exists():
    raise FileExistsError("This run already exists. Choose a fresh RUN_NAME in 1A.")
OUTPUT_ROOT.mkdir(parents=True, exist_ok=True)

acquisition = {
    "cell": "2P Acquire coupled FDE", "dataset": str(DATASET_DIR),
    "settings": settings.configuration(), "section_model": section_model.configuration(),
    "mode_api": str(loaded_api), "completed_nodes": [], "status": "started",
}
started = time.perf_counter()
updater = None

def record_node(node):
    product = dreme.DataUpdater.calc_overlap(node.E, node.H, node.grid.y, node.grid.z, 0)
    max_error = float(np.max(np.abs(np.diag(product) - 1)))
    if max_error > 1e-9:
        raise ValueError(f"Coupled normalization check failed at {node.node}: {max_error:g}")
    acquisition["completed_nodes"].append({
        "gap_nm": node.node[0] * 1e9, "basis_id": node.basis_id,
        "requested_modes": node.requested_count, "returned_modes": node.returned_count,
        "field_shape": list(node.E.shape), "normalization_diagonal_max_error": max_error,
        "elapsed_s": time.perf_counter() - started,
    })
    ACQUISITION_LOG.write_text(json.dumps(acquisition, indent=2), encoding="utf-8")
    print(f"gap {node.node[0] * 1e9:g} nm: {node.returned_count}/{node.requested_count} modes, E/H {node.E.shape}")

try:
    updater = dreme.DataUpdater(DATASET_DIR, coupled_settings=settings)
    geometry = dreme.StraightGapCoupler(
        updater, section_model,
        gap_profile=dreme.profile_from_samples(
            [0.0, OFFLINE_LENGTH_UM * 1e-6], [settings.gap_grid[0], settings.gap_grid[-1]],
        ),
        verbose=False,
    )
    try:
        manifest = updater.acquire_coupled(geometry, mode_api=mode_api, on_node=record_node)
    finally:
        updater.close_mode()
    acquisition.update(status="complete", generation_id=manifest["generation_id"],
                       directed_edges=len(manifest["edges"]))
    print(f"Saved {DATASET_DIR} | generation {manifest['generation_id']}")
except Exception as exc:
    acquisition.update(status="error", first_error=f"{type(exc).__name__}: {exc}",
                       traceback=traceback.format_exc())
    raise
finally:
    acquisition["elapsed_s"] = time.perf_counter() - started
    ACQUISITION_LOG.write_text(json.dumps(acquisition, indent=2), encoding="utf-8")
    print(f"{acquisition['status']} after {acquisition['elapsed_s']:.1f} s | {ACQUISITION_LOG}")

## 4 Reopen and propagate offline

Use the same settings and `RUN_NAME` as the saved generation. This reopens only the dataset files and checks uniform-gap complex phase, increasing and decreasing gap propagation, and basis linkage. Pickle files must come from your own run. This checks the software connection; mesh/gap convergence and FDTD comparison are separate validation stages.

In [ ]:
# 4 Reopen and propagate offline -- no MODE session
offline = {"cell": "4 Reopen and propagate offline", "dataset": str(DATASET_DIR), "runs": {}}
try:
    dataset = dreme.DataUpdater(DATASET_DIR, is_testmode=True)
    if dataset.mode is not None:
        raise RuntimeError("Offline dataset unexpectedly owns a MODE session")
    if dataset.coupled_settings != settings:
        raise ValueError("Saved FDE settings differ from 1A. Match the original settings and run name.")
    offline["generation_id"] = dataset.canonical_manifest["generation_id"]
    offline["export_grid_shape"] = [len(dataset.canonical_manifest["export_grid"][axis]) for axis in ("y", "z")]
    offline["neff_interpolation"] = False
    offline["pml_mode_sink"] = PML_MODE_SINK

    lo, hi = settings.gap_grid[0], settings.gap_grid[-1]
    length = OFFLINE_LENGTH_UM * 1e-6
    for label, gaps in (("uniform", (lo, lo)), ("increasing", (lo, hi)), ("decreasing", (hi, lo))):
        coupler = dreme.StraightGapCoupler(
            dataset, section_model, gap_profile=dreme.profile_from_samples([0.0, length], gaps),
            verbose=False,
        )
        eme = dreme.EME(coupler, neff_interpolation=False,
                       stability_config=dreme.EMEStabilityConfig(pml_mode_sink=PML_MODE_SINK))
        start, end = coupler.runtime_basis_at(0), coupler.runtime_basis_at(-1)
        if not len(start.guided_slots):
            raise ValueError(f"{label}: no guided forward supermode at input node {start.node}")
        launch = np.zeros(len(start.guided_slots), dtype=complex)
        launch[0] = 1
        output = dreme.Runner(eme).propagate_lumped_smatrix(launch)
        if output is None or not np.isfinite(output).all():
            raise ValueError(f"{label}: Runner did not return finite complex amplitudes")
        if label == "uniform":
            slot = start.guided_slots[0]
            expected = np.zeros_like(output)
            expected[slot] = np.exp(1j * coupler.output_data["beta"][0, slot] * length)
            np.testing.assert_allclose(output, expected, atol=1e-7, rtol=1e-7)

        # Verify both directed matrices against the actual stored basis and the
        # runtime transforms. Backward columns use the reciprocal conjugate R.
        sections = coupler.output_data["runtime_basis"]
        for i, (a, b) in enumerate(zip(sections, sections[1:])):
            A, B = (np.kron(np.eye(2), s.R) for s in (a, b))  # R is a real sign/permutation.
            ab, ba = dataset.get_overlap(a.node, b.node)
            np.testing.assert_allclose(coupler.output_data["overlap_ab"][i], A.T @ ab @ B, atol=1e-12)
            np.testing.assert_allclose(coupler.output_data["overlap_ba"][i], B.T @ ba @ A, atol=1e-12)

        nodes, distances = coupler.calc_simulation_parameters()
        offline["runs"][label] = {
            "nodes_m": nodes, "distances_m": distances.tolist(),
            "input_basis_id": start.basis_id, "output_basis_id": end.basis_id,
            "input_guided_slots": start.guided_slots.tolist(),
            "output_forward_slots": end.forward_slots.tolist(),
            "output_re_im": np.column_stack((output.real, output.imag)).tolist(),
        }
        print(f"{label}: {start.node} -> {end.node}, {len(nodes)} path nodes, finite complex amplitudes")
    offline["status"] = "complete"
except Exception as exc:
    offline.update(status="error", first_error=f"{type(exc).__name__}: {exc}",
                   traceback=traceback.format_exc())
    raise
finally:
    OUTPUT_ROOT.mkdir(parents=True, exist_ok=True)
    OFFLINE_LOG.write_text(json.dumps(offline, indent=2), encoding="utf-8")
    print(f"{offline['status']} | {OFFLINE_LOG}")